# 09 Analysis: which facets matter (X-learner)

**What this does.** Offline analyses of saved results: no model calls, no financial tables. The main one is Kethan's
X-learner (`facet_uplift.py`, v1 11-13): for each facet it compares pairs with a strong match on that facet against
pairs with a weak one, adjusting for the other facets, and reports the difference in the outcome (the facet's *lift*).
Three questions:

1. **What makes a good comp?** Outcome = Barbara's grade (`dev_293`): which facet matches go with a peer?
2. **What does the selector act on?** Outcome = the Sonnet selector kept the comp (04), on every pair of the current
   B3 lists (2,538 pairs, 373 training targets) and on the same 293 pairs.
3. **Does the selector add anything?** Treatment = the selector's keep, outcome = Barbara's grade, adjusted for all
   B3 facet signals: the selector's lift over what the facets already tell.

Facet signals: B3's per-facet similarity (0.5 embedding + 0.5 TF-IDF, before weighting) of the current variant,
so Fiona's sub-sector is a 12th signal; strong = above the training-fold median. Agent facet scores (05) are a
second, separate signal set where the agent graded the pair (strong = 75/100).

**Read with care.** Lifts are adjusted associations within a reviewed, B3-selected cohort, not causal effects;
facets are correlated, so lifts don't add up. Intervals are a target bootstrap of fixed predictions (too narrow).
The locked `final_holdout_24` pairs are never used.

**Changelog**
- 2026-11-19: created from `v1/scripts/facet_uplift.py`, `v1/FACET_UPLIFT.md` and the two v1 notebooks. Estimator
  unchanged; added `--cohort`, the selector outcome, the selector treatment and the current B3 signals. The v1 runs
  reproduce (section 4). `error_analysis.py` ported as is (it reads the v1 record).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [2]:
import json, facet_uplift as fu
runs = {k: paths.path(v) for k, v in cur["analysis"]["facet_uplift"].items()}
pd.DataFrame({k: {f: json.loads((r / "manifest.json").read_text())[f]
                  for f in ["cohort", "outcome_mode", "features", "pairs", "targets"]} for k, r in runs.items()}).T

,cohort,outcome_mode,features,pairs,targets
human_peer,dev_293,peer,current,293,45
human_rating,dev_293,rating,current,293,45
selector_all,b3_current,selector,current,2538,373
selector_293,dev_293,selector,current,293,45
record_peer,dev_293,peer,record,293,45
record_rating,dev_293,rating,record,293,45


## 2. Inputs (the pinned runs' summaries)

In [3]:
nb.inputs([cur["analysis"]["facet_uplift"][k] + "/facet_summary.csv" for k in runs])

,input,rows,sha256,status
0,outputs-local/v1a/09_analysis/facet_uplift/20261119T021242064364Z/facet_summary.csv,47,ef2d2d17b024,new in v1a
1,outputs-local/v1a/09_analysis/facet_uplift/20261119T021253830100Z/facet_summary.csv,47,65013ae10d5a,new in v1a
2,outputs-local/v1a/09_analysis/facet_uplift/20261119T021304002727Z/facet_summary.csv,46,16162e77eb40,new in v1a
3,outputs-local/v1a/09_analysis/facet_uplift/20261119T021318755980Z/facet_summary.csv,46,e5b3e3c4787a,new in v1a
4,outputs-local/v1a/09_analysis/facet_uplift/20261119T021152291026Z/facet_summary.csv,45,26f6bc4c3c69,new in v1a
5,outputs-local/v1a/09_analysis/facet_uplift/20261119T021120275676Z/facet_summary.csv,45,c8397ec8a7e3,new in v1a


## 3. Results

### 3.1 Facet lift: people's grades vs the selector's keeps

Percentage points: how much more often the pair is a peer (people) or kept (selector) when the facet matches
strongly. `selector (293)` is on the same pairs as `people (293)`; `selector (2538)` is every pair in the current
B3 lists. Complete five-fold analyses only.

In [4]:
t = fu.lift_table({"people (293)": runs["human_peer"], "selector (293)": runs["selector_293"],
                   "selector (2538)": runs["selector_all"]})
lift = (t.xs("lift", axis=1, level=1) * 100).round(1)
lift["people - selector (293)"] = (lift["people (293)"] - lift["selector (293)"]).round(1)
b3 = lift.loc["b3"].sort_values("people (293)", ascending=False)
b3

,people (293),selector (293),selector (2538),people - selector (293)
facet,,,,
primary_services,23.8,8.2,6.0,15.6
end_markets,21.4,10.6,8.9,10.8
customer_segment,20.7,6.7,2.5,14.0
primary_products,15.1,11.8,9.4,3.3
revenue_model,8.9,7.8,7.9,1.1
regulatory_exposure,8.7,7.3,6.2,1.4
specialization,5.3,8.3,4.9,-3.0
offering_description,4.1,14.6,11.3,-10.5
sub_sector,2.5,10.8,13.2,-8.3


In [5]:
ci = (t * 100).round(1)
ci.loc["b3"].sort_values(("people (293)", "lift"), ascending=False)

people (293)             selector (293)             selector (2538)            
                              lift   low  high           lift   low  high            lift   low  high
facet                                                                                                
primary_services              23.8  20.3  27.2            8.2   4.1  12.3             6.0   5.5   6.5
end_markets                   21.4  16.8  25.4           10.6   7.7  13.8             8.9   8.4   9.4
customer_segment              20.7  16.2  24.9            6.7   4.2   9.0             2.5   2.2   2.9
primary_products              15.1  10.5  19.9           11.8   6.9  17.1             9.4   8.8  10.2
revenue_model                  8.9   6.0  12.1            7.8   5.1  10.4             7.9   7.4   8.4
regulatory_exposure            8.7   5.4  11.7            7.3   5.2   9.4             6.2   5.7   6.7
specialization                 5.3   2.0   8.9            8.3   5.6  10.8             4.9   4.5   5.3
offering_description           4.1  -0.5   8.8           14.6  11.7  17.4            11.3  10.7  11.8
sub_sector                     2.5  -1.6   7.4           10.8   7.9  13.7            13.2  12.5  13.8
distribution_channels          1.6  -2.4   5.8            4.4   1.0   7.6             5.0   4.6   5.4
geography_served               0.4  -2.6   3.6            0.2  -1.5   1.7             2.0   1.7   2.4
value_chain_position          -4.3  -7.0  -1.1            2.0  -0.6   4.6             8.2   7.6   8.8

In [6]:
def strong(col, tab=t, src="b3"):
    x = tab.loc[src, col]
    return x[x["low"] > 0].sort_values("lift", ascending=False)
p, s = strong("people (293)"), strong("selector (2538)")
print("B3 facets that go with a human peer (interval above 0):", ", ".join(f"{f} {v*100:+.1f}" for f, v in p["lift"].items()))
print("B3 facets the selector keeps on (all 2,538 pairs):    ", ", ".join(f"{f} {v*100:+.1f}" for f, v in s["lift"].items()))
gap = b3["people - selector (293)"]
print("Under-used by the selector (people lift >> selector lift, same 293 pairs):",
      ", ".join(f"{f} {v:+.1f}" for f, v in gap[gap > 5].items()))
print("Over-used by the selector (selector lift >> people lift):",
      ", ".join(f"{f} {v:+.1f}" for f, v in gap[gap < -5].items()))

B3 facets that go with a human peer (interval above 0): primary_services +23.8, end_markets +21.4, customer_segment +20.7, primary_products +15.1, revenue_model +8.9, regulatory_exposure +8.7, specialization +5.3
B3 facets the selector keeps on (all 2,538 pairs):     sub_sector +13.2, offering_description +11.3, primary_products +9.4, end_markets +8.9, value_chain_position +8.2, revenue_model +7.9, regulatory_exposure +6.2, primary_services +6.0, distribution_channels +5.0, specialization +4.9, customer_segment +2.5, geography_served +2.0
Under-used by the selector (people lift >> selector lift, same 293 pairs): primary_services +15.6, end_markets +10.8, customer_segment +14.0
Over-used by the selector (selector lift >> people lift): offering_description -10.5, sub_sector -8.3, value_chain_position -6.3


### 3.2 Does the selector add to the facets?

The selector's keep as the treatment, adjusted for all 12 B3 signals: the difference in Barbara's grade between kept
and dropped comps that the facets alone don't explain.

In [7]:
rows = []
for name, unit in (("human_peer", "pp of peer probability"), ("human_rating", "rating points (0-2)")):
    s = pd.read_csv(runs[name] / "facet_summary.csv").query("source == 'selector'").iloc[0]
    scale = 100 if name == "human_peer" else 1
    rows.append({"outcome": name, "unit": unit, "lift": round(s.mean_uplift * scale, 2),
                 "low": round(s.uplift_ci_low * scale, 2), "high": round(s.uplift_ci_high * scale, 2),
                 "pairs": int(s.predicted_pairs), "folds ok": int(s.folds_ok)})
pd.DataFrame(rows)

,outcome,unit,lift,low,high,pairs,folds ok
0,human_peer,pp of peer probability,3.99,0.07,7.59,293,5
1,human_rating,rating points (0-2),0.15,0.07,0.22,293,5


In [8]:
f = pd.read_csv(runs["human_peer"] / "pair_features.csv")
pd.crosstab(f["grade"], f["selector_keep"].map({0: "dropped", 1: "kept"}), rownames=["Barbara"], colnames=["selector"], margins=True)

selector,dropped,kept,All
Barbara,,,
0,88,3,91
1,136,42,178
2,0,24,24
All,224,69,293


### 3.3 Agent facet scores (05), same comparison

The agent scored facets only on the 293 pairs it graded, so `selector (2538)` here is in effect the same 293 pairs.
Facets the agent rarely found evidence for have too few strong/weak pairs to estimate (NaN).

In [9]:
a = lift.loc["agent"].drop(columns="people - selector (293)").sort_values("people (293)", ascending=False)
sup = pd.read_csv(runs["human_peer"] / "facet_summary.csv").query("source == 'agent' and rule == 'primary'")
a.join(sup.set_index("facet")[["folds_ok", "predicted_pairs"]], how="outer")

,people (293),selector (293),selector (2538),folds_ok,predicted_pairs
facet,,,,,
business_model,8.3,20.0,21.4,5,279
customer_segment,8.9,16.2,16.1,5,221
distribution_channels,NaN,NaN,NaN,0,0
end_markets,10.7,33.9,31.7,5,234
geography_served,9.0,19.0,12.8,5,134
primary_products,NaN,NaN,NaN,4,179
primary_services,NaN,NaN,NaN,3,77
regulatory_exposure,NaN,NaN,NaN,0,0
revenue_model,NaN,NaN,NaN,0,0


### 3.4 Rating outcome (0/1/2) and the sensitivity rule

Rating: lift in grade points. Sensitivity: upper vs lower third of the facet signal (middle third left out).

In [10]:
r = fu.lift_table({"rating": runs["human_rating"]})
e = fu.lift_table({"people, thirds": runs["human_peer"], "selector 2538, thirds": runs["selector_all"]}, rule="extreme")
pd.concat([r.xs("lift", axis=1, level=1).round(2), (e.xs("lift", axis=1, level=1) * 100).round(1)], axis=1).loc["b3"]\
  .sort_values("rating", ascending=False)

,rating,"people, thirds","selector 2538, thirds"
facet,,,
customer_segment,0.25,21.4,5.8
end_markets,0.24,35.3,12.5
primary_services,0.23,28.2,13.6
primary_products,0.15,28.4,19.2
revenue_model,0.10,14.4,12.6
regulatory_exposure,0.08,9.8,11.0
specialization,0.08,4.9,3.9
sub_sector,0.03,7.9,16.1
distribution_channels,0.02,-0.4,5.2


### 3.5 Error analysis (v1 record)

`error_analysis.py` (Kethan) traces where matching may fail on the v1 Haiku/Sonnet screens of 11-04 and the
round-1 golden set. Rebuilt offline into `outputs-local/v1a/09_analysis/error_analysis/`; inputs are the v1 record.

In [11]:
ea = sorted((paths.out("09_analysis") / "error_analysis").glob("2*"))[-1]
print(ea.name)
print("\n".join((ea / "report.md").read_text(encoding="utf-8").splitlines()[:12]))

20261119T021513460459Z
# Where company matching may be failing

## Findings to act on first

- Only 41/480 historical labels match current Haiku text; just 10 matches are outside company size. Most current matching facets therefore still need verification.
- 2 exact-current, non-size claims carry historical problem labels. See examples below; these are direct review leads rather than inferred causes of pair errors.
- Haiku and Sonnet disagree on 193/2396 matched B3 screen decisions. These are documented hypotheses to inspect, not proof that either model is wrong.
- On the reviewed targets, compare B3's retained peers with the screens' losses before assuming the facets are the main bottleneck. Human direct/adjacent policy matters.

## Scope and evidence

- 36 human-reviewed pairs; 32 with an exact majority across 2 targets.


## 4. Checks

In [12]:
import numpy as np
v1runs = {"record_rating": "20261113T201612982633Z", "record_peer": "20261113T190330202875Z"}
for k, old in v1runs.items():
    a = pd.read_csv(paths.V1_OUTPUTS / "facet_uplift" / old / "facet_summary.csv")
    b = pd.read_csv(runs[k] / "facet_summary.csv").query("source != 'selector'").reset_index(drop=True)
    num = [c for c in a if a[c].dtype.kind == "f"]
    diff = float(np.nanmax(np.abs(a[num].to_numpy() - b[num].to_numpy())))
    checks.add(f"{k} reproduces v1 run {old}", a.drop(columns=num).equals(b[a.columns].drop(columns=num)) and diff < 1e-5,
               f"max difference {diff:.1e}")
for k, r in runs.items():
    m = json.loads((r / "manifest.json").read_text())
    changed = [p for p, h in m["sources"].items() if nb.sha256(ROOT / p) != h]
    checks.add(f"{k}: inputs unchanged since the run", not changed, ", ".join(changed))
f = fu.build_features("dev_293", "peer", "current")
saved = pd.read_csv(runs["human_peer"] / "pair_features.csv", dtype={"target": str, "comp": str})
cols = [c for c in saved if c.startswith(("b3__", "agent__"))]
checks.add("features rebuild from current.toml", f[["target", "comp", "selector_keep"]].equals(saved[["target", "comp", "selector_keep"]])
           and np.nanmax(np.abs(f[cols].to_numpy() - saved[cols].to_numpy())) < 1e-6)
m = json.loads((runs["selector_all"] / "manifest.json").read_text())
checks.add("selector run covers every current B3 pair", m["pairs"] == 2538 and m["targets"] == 373)
import cohorts
try:
    fu.load_pairs("final_holdout_24"); locked = False
except cohorts.LockedCohort:
    locked = True
checks.add("the locked final-check pairs are refused", locked)
leaks = [p.name for p in paths.out("09_analysis").rglob("*.csv")
         if any(w in p.read_text(encoding="utf-8", errors="ignore")[:3000] for w in ("ev_to_ebitda", "EV/EBITDA", "abs % error"))]
checks.add("no financial columns in 09", not leaks, ", ".join(leaks[:5]))
checks.show()

12/12 checks pass


,check,result,detail
0,record_rating reproduces v1 run 20261113T201612982633Z,pass,max difference 3.4e-07
1,record_peer reproduces v1 run 20261113T190330202875Z,pass,max difference 3.0e-07
2,human_peer: inputs unchanged since the run,pass,
3,human_rating: inputs unchanged since the run,pass,
4,selector_all: inputs unchanged since the run,pass,
5,selector_293: inputs unchanged since the run,pass,
6,record_peer: inputs unchanged since the run,pass,
7,record_rating: inputs unchanged since the run,pass,
8,features rebuild from current.toml,pass,
9,selector run covers every current B3 pair,pass,


## 5. History (v1)

- **11-13 (Kethan):** `facet_uplift.py` on Barbara's 293 pairs, v1's 11 B3 signals and the agent's facet scores;
  binary run `20261113T190330202875Z`, rating run `20261113T201612982633Z`, a technical and a stakeholder notebook
  and eight presentation figures (all in `outputs-local/v1/facet_uplift/` and `v1/notebooks/`). Design and caveats:
  `FACET_UPLIFT.md` (copied here).
- **11-17 (Fiona):** `direct_peer_pairs.py standardize` wrote X-learner inputs from Barbara's direct-peer pairs. Those
  pairs are the locked final check (`10_final_holdout`), so that input was deleted (decision 2026-11-18).
- **11-10 (integration study):** 45 one-off `integration_*.py` scripts (scope, growth, judges, holdout comparisons)
  stay in v1 as the record; `README.md` here lists what each one was for.